<a href="https://colab.research.google.com/github/graviraja/100-Days-of-NLP/blob/applications%2Fclassification/applications/classification/sentiment_classification/Sentimix%20using%20LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Initial Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install contractions -q

Dataset can be found [here](https://github.com/gopalanvinay/thesis-vinay-gopalan)

In [ ]:
train_file = '/content/drive/My Drive/train_14k_split_conll.txt'
test_file = '/content/drive/My Drive/dev_3k_split_conll.txt'

### Imports

In [ ]:
import re
import time
import string
import contractions
import numpy as np
import pandas as pd

from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn import metrics

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau

from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
with open(train_file) as f:
    data = f.readlines()

In [ ]:
with open(test_file, 'r') as f:
    test_data = f.readlines()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

### Data Parsing

In [ ]:
def parse_data(data):
    sentences, sentences_info, sentiment = [], [], []
    all_langs = []
    
    single_sentence, single_sentence_info = [], []
    sent = ""
    
    for idx, each_line in enumerate(data):
        line = each_line.strip()
        tokens = line.split('\t')
        num_tokens = len(tokens)
        if num_tokens == 2:
            # add the word
            single_sentence.append(tokens[0])
            # add the word info(lang)
            single_sentence_info.append(tokens[1])
            all_langs.append(tokens[1])
        elif num_tokens == 3 and idx > 0:
            # append the sentence data
            sentences.append(single_sentence)
            sentences_info.append(single_sentence_info)
            sentiment.append(sent)
            sent = tokens[-1]
            # clear the single sentence
            single_sentence = []
            single_sentence_info = []
        # new line after the sentence
        elif num_tokens == 1:
            continue
        else:
            sent = tokens[-1]
    # for the last sentence
    if len(single_sentence) > 0:
        sentences.append(single_sentence)
        sentences_info.append(single_sentence_info)
        sentiment.append(sent)
        
    assert len(sentences) == len(sentences_info) == len(sentiment)
    return sentences, sentences_info, sentiment, all_langs

In [ ]:
sentences, sentences_info, sentiment, all_langs = parse_data(data)

In [ ]:
data[:30]

In [ ]:
test_sentences, test_sentences_info, test_sentiment, test_all_langs = parse_data(test_data)

In [ ]:
len(sentiment)

### Data Exploration

In [ ]:
sns.countplot(all_langs)

In [ ]:
sns.countplot(sentiment)

In [ ]:
set(sentiment)

In [ ]:
sent_num_tokens = [len(sent) for sent in sentences]

In [ ]:
plt.figure(figsize=(15, 5))
sns.countplot(sent_num_tokens)

In [ ]:
sentences[10]

In [ ]:
sentences_info[10]

In [ ]:
sentiment[0]

### Data Cleaning

In [ ]:
url_pattern = r'https(.*)/\s\w+'
special_chars = r'[_?\*\[\]\(\)&?]'
names_with_numbers = r'([A-Za-z]+)\d{3,}'
apostee = r"([\w]+)\s'\s([\w]+)"
names = r"@[\s]*[\w]+[\s]*[_]+[\s]*[\w]+|@[\s]*[\w]+"

def preprocess_data(sentence_tokens):
    sentence = " ".join(sentence_tokens)
    sentence = " " + sentence
    # remove rt and ? from string
    sentence = sentence.replace(" RT ", "")
    sentence = sentence.replace("?", "")
    # replace apostee
    sentence = sentence.replace("?", "'")
    # replace names
    sentence = re.sub(re.compile(names), " ", sentence)
    # remove special chars
    # sentence = re.sub(re.compile(special_chars), "", sentence)
    # remove urls
    sentence = re.sub(re.compile(url_pattern), "", sentence)
    ## remove duplicate characters
    # sentence = re.sub(r"(.)\1{3,}", r'\1', sentence)
    # combine only ' related words => ... it ' s ... -> ... it's ...
    sentence = re.sub(re.compile(apostee), r"\1'\2", sentence)
    # fix contractions
    sentence = contractions.fix(sentence)
    # replace names ending with numbers with only names (remove numbers)
    sentence = re.sub(re.compile(names_with_numbers), r" ", sentence)
    ## consider only printable chars (many greek, urdu, hindi chars are there)
    # sentence = [ch for ch in sentence if ch in string.printable]
    # sentence = "".join(sentence).strip()
    sentence = " ".join(sentence.split()).strip()
    return sentence

In [ ]:
" ".join(sentences[1]), sentiment[1]

In [ ]:
preprocess_data(sentences[1])

In [ ]:
" ".join(sentences[29]), sentiment[29]

In [ ]:
preprocess_data(sentences[29])

In [ ]:
" ".join(sentences[10]), sentiment[10]

In [ ]:
preprocess_data(sentences[10])

In [ ]:
%%time
processed_sentences = []

for sent in sentences:
    processed_sentences.append(preprocess_data(sent))

In [ ]:
%%time
test_data = []

for sent in test_sentences:
    test_data.append(preprocess_data(sent))

In [ ]:
sentiment_mapping = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

In [ ]:
labels = [sentiment_mapping[sent] for sent in sentiment]
test_label = [sentiment_mapping[sent] for sent in test_sentiment]

### Train-Val-Test Splits

In [ ]:
train_data, val_data, train_label, val_label = train_test_split(processed_sentences, labels, test_size=0.2)

In [ ]:
len(train_data), len(val_data), len(test_data)

### Train-Val-Test Distributions

In [ ]:
sns.countplot(train_label)
plt.xlabel('Training Data', fontsize=16)

In [ ]:
sns.countplot(val_label)
plt.xlabel('Validation Data', fontsize=16)

In [ ]:
sns.countplot(test_label)
plt.xlabel('Testing Data', fontsize=16)

### Vocabulary

In [ ]:
class Vocabulary(object):
    def __init__(self):
        self.word2idx = {}
        self.idx2word = {}
        self.idx = 0

    def add_word(self, word):
        if not word in self.word2idx:
            self.word2idx[word] = self.idx
            self.idx2word[self.idx] = word
            self.idx += 1

    def __call__(self, word):
        if not word in self.word2idx:
            return self.word2idx['<unk>']
        return self.word2idx[word]

    def __len__(self):
        return len(self.word2idx)

In [ ]:
def build_vocab(sentences, threshold=15):
    """Build a simple vocabulary wrapper."""
    counter = Counter()
    for i, sent in enumerate(sentences):
        counter.update(sent.split())

    # If the word frequency is less than 'threshold', then the word is discarded.
    words = [word for word, cnt in counter.items() if cnt >= threshold]

    # Create a vocab wrapper and add some special tokens.
    vocab = Vocabulary()
    vocab.add_word('<pad>')
    vocab.add_word('<start>')
    vocab.add_word('<end>')
    vocab.add_word('<unk>')

    # Add the words to the vocabulary.
    for i, word in enumerate(words):
        vocab.add_word(word)
    return vocab

In [ ]:
vocab = build_vocab(train_data)

In [ ]:
len(vocab)

### Dataset Wrapper

In [ ]:
class SentiMixDataSet(Dataset):
    def __init__(self, inputs, labels):
        self.sentences = inputs
        self.labels = labels
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, item):
        sentence = self.sentences[item]
        sentiment = int(self.labels[item])
        
        tokens = []
        tokens.append(vocab('<start>'))
        for tok in sentence.split():
            tokens.append(vocab(tok))
        tokens.append(vocab('<end>'))
        return torch.LongTensor(tokens), sentiment

In [ ]:
train_dataset = SentiMixDataSet(train_data, train_label)
val_dataset = SentiMixDataSet(val_data, val_label)
test_dataset = SentiMixDataSet(test_data, test_label)

In [ ]:
# sample check
train_dataset[0]

### DataLoaders

In [ ]:
def collate_fn(data):
    data.sort(key=lambda x: len(x[0]), reverse=True)
    sentences, sentiments = zip(*data)
    
    sent_lengths = [len(sent) for sent in sentences]
    inputs = torch.zeros((len(sentences), max(sent_lengths)), dtype=torch.long)
    labels = torch.zeros(len(sentences), dtype=torch.long)

    for i, sent in enumerate(sentences):
        end = sent_lengths[i]
        inputs[i, :end] = sent[:end]
        labels[i] = sentiments[i]

    return inputs, sent_lengths, labels

In [ ]:
BATCH_SIZE = 16

train_data_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn, shuffle=True)
valid_data_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)
test_data_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)

In [ ]:
sample = next(iter(train_data_loader))
sample[0].shape, len(sample[1]), sample[2].shape

### RNN Model

In [ ]:
class RNNModel(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, output_dim, num_layers, dropout=0.4):
        super().__init__()

        self.hid_dim = hid_dim
        self.num_layers = num_layers
        self.embedding = nn.Embedding(input_dim, emb_dim)

        self.rnn = nn.GRU(
            emb_dim,
            hid_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        self.fc = nn.Linear(hid_dim, 50)
        self.out = nn.Linear(50, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs, input_lengths):
        # inputs => [batch_size, seq_len]
        # input_lengths => [batch_size]

        embedded = self.dropout(self.embedding(inputs))

        packed_input = nn.utils.rnn.pack_padded_sequence(embedded, input_lengths, batch_first=True)
        _, hidden = self.rnn(packed_input)

        hidden = hidden.view(self.num_layers, 2, -1, self.hid_dim)
        final_forward_hidden = hidden[-1, -2, :, :]
        final_backward_hidden = hidden[-1, -1, :, :]
        # final_*_hidden => [batch_size, hidden_dim]

        combined = final_forward_hidden + final_backward_hidden
        combined = self.dropout(combined)
        # combined => [batch_size, hidden_dim]

        intermediate = F.relu(self.fc(combined))
        intermediate = self.dropout(intermediate)
        logits = self.out(intermediate)
        # logits => [batch_size, output_dim]

        return logits

### Model Configurations

In [ ]:
input_dim = len(vocab)
output_dim = 3
emb_dim = 200
hid_dim = 100
num_layers = 2
NUM_EPOCHS = 20
model_path = "rnn.pt"

In [ ]:
model = RNNModel(input_dim, emb_dim, hid_dim, output_dim, num_layers)

In [ ]:
model.to(device)

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"The model has {count_parameters(model)} trainable parameters")


### Loss Criterion & Optimizer

In [ ]:
lr = 1e-4
min_lr = 3e-5
lr_decay=0.5
lr_patience=2

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)
scheduler = ReduceLROnPlateau(optimizer, 'min', lr_decay, lr_patience, verbose=True, min_lr=min_lr)

### Training Method

In [ ]:
def train(iterator, clip=2.0):
    epoch_loss = 0
    model.train()
    for batch in iterator:
        sentences = batch[0].to(device)
        sentence_lengths = batch[1]
        targets = batch[2].to(device)

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(sentences, sentence_lengths)
        loss = criterion(outputs, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        epoch_loss += loss.item()
    return epoch_loss / len(iterator)

### Validation Method

In [ ]:
def categorical_accuracy(preds, y):
    """
    Returns accuracy per batch, i.e. if you get 8/10 right, this returns 0.8, NOT 8
    """
    max_preds = preds.argmax(dim = 1, keepdim = True) # get the index of the max probability
    correct = max_preds.squeeze(1).eq(y)
    return correct.sum() / torch.FloatTensor([y.shape[0]]).to(device)

In [ ]:
def evaluate(iterator):
    epoch_loss = 0
    model.eval()
    epoch_acc = 0

    with torch.no_grad():
        for batch in iterator:
            sentences = batch[0].to(device)
            sentence_lengths = batch[1]
            targets = batch[2].to(device)
            logits = model(sentences, sentence_lengths)
            # logits => [batch_size, num_labels]

            loss = criterion(logits, targets)
            acc = categorical_accuracy(logits, targets)
            epoch_acc += acc.item()
            epoch_loss += loss.item()
    
    return epoch_loss / len(iterator), epoch_acc / len(iterator)

In [ ]:
def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = elapsed_time - (elapsed_mins * 60)
    return elapsed_mins, elapsed_secs

### Model Training

In [ ]:
best_valid_loss = float('inf')
for epoch in range(NUM_EPOCHS):
    start_time = time.time()
    train_loss = train(train_data_loader)
    val_loss, val_acc = evaluate(valid_data_loader)
    end_time = time.time()
    scheduler.step(val_loss)

    epoch_mins, epoch_secs = epoch_time(start_time, end_time)
    print(f"Epoch: {epoch + 1:02} | Time: {epoch_mins}m {epoch_secs:.2f}s")
    print(f"\tTrain Loss: {train_loss:.3f} | Val Loss: {val_loss:.3f} | Val Acc: {val_acc:.3f}")
    
    if val_loss < best_valid_loss:
        best_valid_loss = val_loss
        torch.save(model.state_dict(), model_path)

In [ ]:
model.load_state_dict(torch.load(model_path))

### Evaluation

In [ ]:
def cal_metrics(model, data_loader):
    model.eval()
    fin_outputs = []
    fin_targets = []

    with torch.no_grad():
        for batch in data_loader:
            sentences = batch[0].to(device)
            sentence_lengths = batch[1]
            targets = batch[2].to(device)

            predictions = model(sentences, sentence_lengths)
            # predictions => [batch_size, num_labels]

            outputs = predictions.max(dim=1)[1]
            fin_targets.extend(targets.detach().cpu().numpy().tolist())
            fin_outputs.extend(outputs.detach().cpu().numpy().tolist())

    assert len(fin_outputs) == len(fin_targets)

    cf = metrics.classification_report(fin_targets, fin_outputs)
    print(cf)


In [ ]:
cal_metrics(model, test_data_loader)